In [ ]:
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatPromptTemplate

)
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from typing import List
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter


class ProductCatalogExtraction(BaseModel):
    """Model for product catalog extraction"""
    product_name: List[str] = Field(..., description="List of product names")
    product_description: List[str] = Field(..., description="List of product descriptions")

class ProductCatalogExtractionPrompt:
    """Prompt template for extracting product catalog information"""
    
    def __init__(self) -> None:

        self.system_message = SystemMessagePromptTemplate.from_template(
            """
            You are a helpful assistant for extracting product catalog information.
            """
        )

        self.human_message = HumanMessagePromptTemplate.from_template(
            """
            Extract the product names and descriptions from the following text:
            {text}

            output should be in the format:
            {{
                "product_name": ["Product 1", "Product 2", ...],
                "product_description": ["Description 1", "Description 2", ...]
            }}
            """
        )

    def get_prompt(self) -> ChatPromptTemplate:
        """Returns the complete prompt template"""
        return ChatPromptTemplate.from_messages(
            [
                self.system_message,
                self.human_message
            ]
        )
    
class ProductCatalogSummarizationModel(BaseModel):
    """Model for summarizing product catalog information"""
    # ProductList: List[str] = Field(..., description="List of product names")
    Summary: List[str] = Field(..., description="List of summaries for each product")
    
class ProductCatalogSummarizationPrompt:
    """Prompt template for summarizing product catalog information"""
    
    def __init__(self) -> None:

        self.system_message = SystemMessagePromptTemplate.from_template(
            """
            You are a {role} for summarizing product catalog information.
            """
        )

        self.human_message = HumanMessagePromptTemplate.from_template(
            """
            Summarize the following List of product catalog information, where i provided data column name rows separated by new line.
            Each row represents a product and its features.
            your task is to summarize the product catalog information based on the provided features. Don't replicate the input text in the output.
            if particular features are missing, then just summarize the available features without throwing an error or returning empty values.
            input:
            column names(Features): 
            {text}
            row values(Values): 
            {csv_data}

            output should be in the format:
            {{
                Summary: [Product 1 summary, Product 2 summary, ...]
            }}
            """
        )

    def get_prompt(self) -> ChatPromptTemplate:
        """Returns the complete prompt template"""
        return ChatPromptTemplate.from_messages(
            [
                self.system_message,
                self.human_message
            ]
        )

In [80]:
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI, ChatGoogleGenerativeAIError
import os
from dotenv import load_dotenv

load_dotenv("../.env")


True

In [81]:
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY is not set in the environment variables.")
print("Google API Key is set.")
print(api_key[:10])

Google API Key is set.
AIzaSyBoA_


In [82]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    api_key=api_key,
    temperature=0.7
)

In [95]:
from langchain_core.runnables import RunnableSequence
prompt1 = ProductCatalogExtractionPrompt().get_prompt()
output_parser = PydanticOutputParser(pydantic_object=ProductCatalogSummarizationModel)
prompt2 = ProductCatalogSummarizationPrompt().get_prompt()


runnable = RunnableSequence(
    prompt2
    | llm
    # | prompt2
    # | llm
    | output_parser
)

In [96]:
data = pd.read_csv("../myntra_products_catalog.csv")

In [97]:
columns = ", ".join(data.columns.tolist())

In [98]:
data_csv = data.head(100).to_csv(header=False)

In [99]:
columns

'ProductID, ProductName, ProductBrand, Gender, Price (INR), NumImages, Description, PrimaryColor'

In [100]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,
    chunk_overlap=500,
)

In [101]:
chunks = text_splitter.split_text(data_csv)

In [102]:
def process_chunk(chunk: str) -> None:
    split_pos = chunk.rfind("\n")

    if split_pos != -1:
        part1 = chunk[:split_pos]      # everything before last '\n'
        part2 = chunk[split_pos+1:]    # everything after last '\n'
    else:
        part1 = chunk
        part2 = ""

    return part1, part2

In [104]:
part2

'99,10003299,Gini and Jony Boys Red & Grey Melange Solid Varsity Jacket with Applique Detail,Gini and Jony,Boys,649,3,"Red and Grey Melange solid varsity jacket, has a stand collar, snap button closure, long sleeves, straight hem, unlined", Red'

In [106]:
# Optimized processing with smaller batches and progress tracking
import time
from tqdm import tqdm

print(f"Total chunks to process: {len(chunks)}")
print("Starting optimized processing...")

# Process only first few chunks for testing
max_chunks = 1  # Limit for testing - increase this as needed
processed_count = 0

content = []

part2 = ""  # Initialize part2 to store remaining text
for i, chunk in enumerate(chunks[:max_chunks]):
    try:
        print(f"\n🔄 Processing chunk {i+1}/{min(max_chunks, len(chunks))}...")
        
        # Add timeout and retry logic
        start_time = time.time()

        chunk = part2 + chunk  # Combine with any remaining text from previous chunk
        part1, part2 = process_chunk(chunk)  # Process the chunk to split it

        print(part1)

        result = runnable.invoke({
            "role": "fashion retailassistant",
            "text": columns,
            "csv_data": part1,
        })

        print("✅ Chunk processed successfully!")
        print(result.content)
        content.extend(result)
            
    except KeyboardInterrupt:
        print("\n⏹️ Processing stopped by user")
        break
    except Exception as e:
        print(f"❌ Error processing chunk {i+1}: {str(e)}")
        continue

print(f"\n✅ Processing complete! Processed {processed_count} products from {i+1} chunks.")
print(f"📊 Collected {len(all_product_names)} product entries for DataFrame")

Total chunks to process: 6
Starting optimized processing...

🔄 Processing chunk 1/1...
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolley Bag,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, secured with a TSA lockOne handle on the top and one on the side, has a trolley with a retractable handle on the top and four corner mounted inline skate wheelsOne main zip compartment, zip lining, two compression straps with click clasps, one zip compartment on the flap with three zip pocketsWarranty: 5 yearsWarranty provided by Brand Owner / Manufacturer", Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure Custom Made Kurta Set with Jacket,EthnoVogue,Women,5810,7,"Beige & Grey made to measure kurta with churidar and dupattaBeige made to measure calf length kurta, has a V-neck, three-quarter sleeves, lightly padded on bust, flared hem, concealed zip closureGrey solid made to measure churidar, drawstring closureGrey net sequined dupatta, has printed tapingWhat is 

In [16]:
# Create DataFrame from collected results
import pandas as pd
from datetime import datetime

if all_product_names and all_summaries:
    # Create DataFrame
    results_df = pd.DataFrame({
        'product_name': all_product_names,
        'summary': all_summaries,
    })
    
    print("📊 DataFrame created successfully!")
    print(f"Shape: {results_df.shape}")
    print("\n📋 DataFrame preview:")
    print(results_df.head())
    print("\n Shape of the DataFrame:", results_df.shape)
    
else:
    print("❌ No data collected. Please run the processing cell first.")

📊 DataFrame created successfully!
Shape: (34, 2)

📋 DataFrame preview:
                                        product_name  \
0  DKNY Unisex Black & Grey Printed Medium Trolle...   
1  EthnoVogue Women Beige & Grey Made to Measure ...   
2  SPYKAR Women Pink Alexa Super Skinny Fit High-...   
3  Raymond Men Blue Self-Design Single-Breasted B...   
4  Parx Men Brown & Off-White Slim Fit Printed Ca...   

                                             summary  
0  DKNY unisex black and grey printed medium trol...  
1  EthnoVogue women's beige and grey custom-made ...  
2  SPYKAR Women's pink super skinny fit high-rise...  
3  Raymond Men's blue self-design single-breasted...  
4  Parx Men's brown and off-white slim fit printe...  

 Shape of the DataFrame: (34, 2)


In [94]:
content

['`',
 '`',
 '`',
 'j',
 's',
 'o',
 'n',
 '\n',
 '{',
 '\n',
 ' ',
 ' ',
 ' ',
 ' ',
 '"',
 'S',
 'u',
 'm',
 'm',
 'a',
 'r',
 'y',
 '"',
 ':',
 ' ',
 '[',
 '\n',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 ' ',
 '"',
 'A',
 ' ',
 'b',
 'l',
 'a',
 'c',
 'k',
 ' ',
 'a',
 'n',
 'd',
 ' ',
 'g',
 'r',
 'e',
 'y',
 ' ',
 'p',
 'r',
 'i',
 'n',
 't',
 'e',
 'd',
 ' ',
 'm',
 'e',
 'd',
 'i',
 'u',
 'm',
 ' ',
 't',
 'r',
 'o',
 'l',
 'l',
 'e',
 'y',
 ' ',
 'b',
 'a',
 'g',
 ' ',
 'f',
 'r',
 'o',
 'm',
 ' ',
 'D',
 'K',
 'N',
 'Y',
 ',',
 ' ',
 'd',
 'e',
 's',
 'i',
 'g',
 'n',
 'e',
 'd',
 ' ',
 'f',
 'o',
 'r',
 ' ',
 'u',
 'n',
 'i',
 's',
 'e',
 'x',
 ' ',
 'u',
 's',
 'e',
 ',',
 ' ',
 'f',
 'e',
 'a',
 't',
 'u',
 'r',
 'e',
 's',
 ' ',
 'a',
 ' ',
 'T',
 'S',
 'A',
 ' ',
 'l',
 'o',
 'c',
 'k',
 ',',
 ' ',
 'r',
 'e',
 't',
 'r',
 'a',
 'c',
 't',
 'a',
 'b',
 'l',
 'e',
 ' ',
 'h',
 'a',
 'n',
 'd',
 'l',
 'e',
 ',',
 ' ',
 'i',
 'n',
 'l',
 'i',
 'n',
 'e',
 ' ',
 's',
 'k',
 